# Mixed Precision Evolution with OpenEvolve

This tutorial demonstrates how to use OpenEvolve to optimize both:
1. **Trotter splitting sequences** (e.g., first-order, second-order, custom orderings)
2. **Precision configurations** (FP16, FP32, FP64) for each Hamiltonian term

The goal is to minimize computational cost (FLOPs) while maintaining low infidelity.

## 1. Setup: Import Libraries and Generate Hamiltonian

In [3]:
import numpy as np
import scipy
import trovolut as trv
from pathlib import Path
from trovolut.backends.gpu_mp_emulator.dynamics import (
    Precision, PrecisionConfig, ExpTerm, TrotterSimulator,
    SecondOrderTrotter, PrecisionCaster
)
from trovolut.backends.gpu_mp_emulator.metrics import CostAnalyzer, print_cost_report
from trovolut import qm
from rdkit import Chem
from rdkit.Chem import AllChem
import warnings
warnings.filterwarnings("ignore")

### Generate DMDS Molecular Hamiltonian

We'll use a dimethyl disulfide (DMDS) molecule as our test system. This gives us a realistic molecular Hamiltonian with multiple Pauli terms that can be independently configured with different precisions.

In [1]:
# Generate DMDS molecule
mol = Chem.AddHs(Chem.MolFromSmiles("CSSC"))
AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
AllChem.UFFOptimizeMolecule(mol)

# Extract XYZ coordinates
conf = mol.GetConformer()
atoms = [a.GetSymbol() for a in mol.GetAtoms()]
xyz = (
    f"{len(atoms)}\ndmds generated by RDKit\n"
    + "\n".join(
        f"{s} {conf.GetAtomPosition(i).x:.8f} "
        f"{conf.GetAtomPosition(i).y:.8f} "
        f"{conf.GetAtomPosition(i).z:.8f}"
        for i, s in enumerate(atoms)
    )
)

# Build the electronic Hamiltonian with minimal active space (4 qubits)
H_fermion = qm.ElectronicHamiltonian.from_xyz(
    xyz_source=xyz,
    basis="sto-3g",
    charge=0,
    spin_mult=1,
    use_gpu4pyscf=False,
    freeze_core=True,
    n_core_keep=1,  # Just HOMO
    n_virt_keep=1,  # Just LUMO -> 2 spatial orbs = 4 qubits (dim 16)
)

# Convert to qubit Hamiltonian via Jordan-Wigner
H_qubit = H_fermion.to_qubits()
qmeta = H_qubit.decompose()

# Get Hamiltonian terms (excludes identity as it's just a global phase)
hamiltonian_terms = H_qubit.terms()

# Get info about the Hamiltonian
meta = H_fermion.meta
n_qubits = meta['n_spin_orb']
n_terms = len(hamiltonian_terms)

print(f"System: DMDS molecule (Dimethyl Disulfide)")
print(f"Active space: {meta['n_act_spatial']} spatial orbitals, {meta['n_electrons']} electrons")
print(f"Number of qubits: {n_qubits}")
print(f"Hilbert space dimension: {2**n_qubits}")
print(f"Number of Pauli terms: {len(qmeta['terms'])}")
print(f"Number of Hamiltonian terms (excluding identity): {n_terms}")
print(f"\nFirst 5 terms:")
for i, term in enumerate(qmeta['terms'][:5]):
    print(f"  {i}: coeff={term['coeff']:.6f}, pauli={term['pauli']}")

NameError: name 'Chem' is not defined

## 2. Define Optimization Constraints

We want to minimize computational cost (FLOPs) while keeping the simulation error (infidelity) below a threshold.

In [3]:
# Create a custom FLOPS constraint
# We'll extract FLOPS from the cost analyzer in the evaluator
from trovolut.core.base_constraints import Constraint, Group
import math

class FLOPsConstraint(Constraint):
    """Constraint on total floating point operations."""
    
    def __init__(self, max_flops: float = 1e12, weight: float = 1.0, 
                 hard: bool = False, hard_penalty: float = -100.0, 
                 score_fn: str = "exponential"):
        super().__init__("flops", weight, hard, hard_penalty)
        self.max_flops = max_flops
        self.score_fn = score_fn
    
    def extract(self, m: dict) -> float:
        return m.get("total_flops", float("inf"))
    
    def score(self, v: float) -> float:
        if self.score_fn == "exponential":
            return math.exp(-v / self.max_flops)
        elif self.score_fn == "inverse":
            return 1.0 / (1.0 + v / self.max_flops)
        elif self.score_fn == "linear":
            return max(0, 1.0 - v / self.max_flops)
        return 0.0
    
    def is_satisfied(self, v: float) -> bool:
        return v <= self.max_flops
    
    def to_prompt(self) -> str:
        if self.hard:
            return f"Total FLOPs MUST stay below {self.max_flops:.2e} (hard requirement). Use lower precision (FP16/FP32), fewer steps, efficient decompositions."
        return f"Minimize FLOPs (target: <{self.max_flops:.2e}, weight: {self.weight}). Consider: FP16 for small terms, taylor/krylov for dense terms, optimal step count."
    
    def formula(self) -> str:
        if self.score_fn == "exponential":
            base = f"exp(-flops/{self.max_flops:.2e})"
        elif self.score_fn == "inverse":
            base = f"1 / (1 + flops/{self.max_flops:.2e})"
        else:
            base = f"max(0, 1 - flops/{self.max_flops:.2e})"
        
        if self.hard:
            return f"score = {base} · 𝟙(flops ≤ {self.max_flops:.2e}) + ({self.hard_penalty}) · 𝟙(flops > {self.max_flops:.2e})"
        if self.weight != 1.0:
            return f"score = {base}  (×{self.weight} weight)"
        return f"score = {base}"

# Define constraints
flops_constraint = FLOPsConstraint(
    max_flops=1e12,  # 1 TFLOPs
    hard=False,
    score_fn="exponential",
    weight=2.0  # Prioritize reducing FLOPs
)

error_constraint = trv.eval.Error(
    max_error=0.01,  # 1% infidelity
    hard=True,
    hard_penalty=-1000.0
)

constraints = Group([flops_constraint, error_constraint])
print(f"Optimization constraints:")
print(constraints.formula())

# Define a custom metric function to extract FLOPs from the simulator
def compute_flops(hams, t, psi0, simulator_fn) -> float:
    """Extract total FLOPs from simulator using CostAnalyzer."""
    from trovolut.eval.mp_evals import CostAnalyzer
    import inspect
    
    # Get simulator instance from the function call
    # We need to run the function to get the simulator
    result = simulator_fn(hams, t, psi0)
    
    # Extract simulator from the function's globals if available
    # This is a workaround - ideally simulator_fn would return (result, simulator)
    # For now, return a placeholder that will be calculated in the evaluator
    return 0  # Placeholder

Optimization constraints:
Hard guards: (error satisfied)
  → If any violated: score = penalty (-1000.0)
Soft score: (2.0·s_flops) / 2.0


## 3. Setup Error Measurement

Define how to compute the exact evolution for comparison.

In [4]:
# Define helper function to generate DMDS Hamiltonian
# This will be embedded in the evaluator by build_evaluator()
def get_dmds_hamiltonian():
    """Generate DMDS molecular Hamiltonian (self-contained for evaluator)."""
    from trovolut import qm
    from rdkit import Chem
    from rdkit.Chem import AllChem
    
    # Generate DMDS molecule
    mol = Chem.AddHs(Chem.MolFromSmiles("CSSC"))
    AllChem.EmbedMolecule(mol, AllChem.ETKDGv3())
    AllChem.UFFOptimizeMolecule(mol)
    
    # Extract XYZ
    conf = mol.GetConformer()
    atoms = [a.GetSymbol() for a in mol.GetAtoms()]
    xyz = (
        f"{len(atoms)}\ndmds generated by RDKit\n"
        + "\n".join(
            f"{s} {conf.GetAtomPosition(i).x:.8f} "
            f"{conf.GetAtomPosition(i).y:.8f} "
            f"{conf.GetAtomPosition(i).z:.8f}"
            for i, s in enumerate(atoms)
        )
    )
    
    # Build Hamiltonian
    H_fermion = qm.ElectronicHamiltonian.from_xyz(
        xyz_source=xyz,
        basis="sto-3g",
        charge=0,
        spin_mult=1,
        use_gpu4pyscf=False,
        freeze_core=True,
        n_core_keep=1,
        n_virt_keep=1,
    )
    
    # Convert to qubit (NO sparse parameter!)
    H_qubit = H_fermion.to_qubits()
    hams_terms = H_qubit.terms()
    hams = [term['coef'] * term['matrix'] for term in hams_terms]
    
    return hams

def setup():
    """Return inputs for the candidate function: (hams, t, psi0)."""
    import numpy as np
    
    hams = get_dmds_hamiltonian()
    
    # Random initial state
    dim = hams[0].shape[0]
    np.random.seed(42)
    psi0 = np.random.randn(dim) + 1j * np.random.randn(dim)
    psi0 /= np.linalg.norm(psi0)
    
    return hams, 1.0, psi0

def exact_evolution(hams, t, psi0):
    """Compute exact evolution for reference."""
    import scipy.linalg
    
    H_total = sum(hams)
    # Convert sparse to dense if needed
    if hasattr(H_total, 'toarray'):
        H_total = H_total.toarray()
    return scipy.linalg.expm(-1j * H_total * t) @ psi0

# Configure error measurement
constraints.config_error(
    exact_fn=exact_evolution,
    setup_fn=setup
).entry_point("run_mp_trotter")

print("\nError measurement configured.")
print(f"Setup will use DMDS Hamiltonian.")


Error measurement configured.
Setup will use DMDS Hamiltonian.


## 4. Setup Error Measurement (IMPORTANT: Do this AFTER generating Hamiltonian!)

Define how to compute the exact evolution for comparison. The setup and exact functions will be embedded in the evaluator.

In [5]:
# Custom evaluator function that measures FLOPs
def create_custom_evaluator(evaluator_path: str):
    """Create a custom evaluator that computes FLOPs using CostAnalyzer."""
    
    code = '''
import time
import importlib.util
import numpy as np
import scipy.linalg
from trovolut.core.base_constraints import Group
from trovolut.eval.mp_evals import CostAnalyzer

# Recreate the FLOPsConstraint class
import math
from trovolut.eval.constraints import Constraint

class FLOPsConstraint(Constraint):
    """Constraint on total floating point operations."""
    
    def __init__(self, max_flops: float = 1e12, weight: float = 1.0, 
                 hard: bool = False, hard_penalty: float = -100.0, 
                 score_fn: str = "exponential"):
        super().__init__("flops", weight, hard, hard_penalty)
        self.max_flops = max_flops
        self.score_fn = score_fn
    
    def extract(self, m: dict) -> float:
        return m.get("total_flops", float("inf"))
    
    def score(self, v: float) -> float:
        if self.score_fn == "exponential":
            return math.exp(-v / self.max_flops)
        elif self.score_fn == "inverse":
            return 1.0 / (1.0 + v / self.max_flops)
        elif self.score_fn == "linear":
            return max(0, 1.0 - v / self.max_flops)
        return 0.0
    
    def is_satisfied(self, v: float) -> bool:
        return v <= self.max_flops
    
    def to_prompt(self) -> str:
        if self.hard:
            return f"Total FLOPs MUST stay below {self.max_flops:.2e} (hard requirement)."
        return f"Minimize FLOPs (target: <{self.max_flops:.2e}, weight: {self.weight})."
    
    def formula(self) -> str:
        if self.score_fn == "exponential":
            base = f"exp(-flops/{self.max_flops:.2e})"
        else:
            base = f"max(0, 1 - flops/{self.max_flops:.2e})"
        if self.hard:
            return f"score = {base} · 𝟙(flops ≤ {self.max_flops:.2e})"
        return f"score = {base}"

# Setup function from notebook
'''
    
    # Add the setup and exact functions
    code += f'''
{inspect.getsource(setup)}

{inspect.getsource(exact_evolution)}

# Recreate constraints
flops_constraint = FLOPsConstraint(
    max_flops=1e12,
    hard=False,
    score_fn="exponential",
    weight=2.0
)

import trovolut as trv
error_constraint = trv.eval.Error(
    max_error=0.01,
    hard=True,
    hard_penalty=-1000.0
)

constraints = Group([flops_constraint, error_constraint])

def evaluate(program_path: str) -> dict:
    """Custom evaluator that measures error and FLOPs."""
    # Load candidate module
    spec = importlib.util.spec_from_file_location("candidate", program_path)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    
    candidate_fn = getattr(module, "run_mp_trotter")
    
    metrics = {{}}
    
    # Setup inputs
    inputs = setup()
    hams, t, psi0 = inputs
    
    # Run candidate and time it
    start = time.perf_counter()
    result = candidate_fn(hams, t, psi0)
    metrics["time_seconds"] = time.perf_counter() - start
    
    # Compute error
    reference = exact_evolution(hams, t, psi0)
    metrics["mean_error"] = 1.0 - abs(np.vdot(reference, result)) ** 2
    
    # Estimate FLOPs by analyzing the candidate code
    # This is a rough estimate based on typical parameters
    # OpenEvolve will optimize the actual implementation
    # For now, use a placeholder that scales with state dimension
    dim = len(psi0)
    n_terms = len(hams)
    # Rough estimate: n_steps * n_terms * cost_per_term
    # Each expm costs ~13 * 2 * dim^3 FLOPs (Padé approximation)
    n_steps = 100  # default
    flops_per_expm = 13 * 2 * (dim ** 3)
    estimated_flops = n_steps * n_terms * flops_per_expm
    metrics["total_flops"] = estimated_flops
    
    return constraints.evaluate(metrics)
'''
    
    with open(evaluator_path, 'w') as f:
        f.write(code)
    
    print(f"Created custom evaluator at: {evaluator_path}")

print("Custom evaluator function defined.")

Custom evaluator function defined.


## 5. Initial Program: Mixed Precision Trotter

This is the starting point for OpenEvolve. It uses a simple mixed precision configuration that OpenEvolve will optimize.

**What OpenEvolve can optimize:**
- The Trotter splitting sequence (order of terms, symmetrization)
- The precision used for each term (FP16, FP32, FP64)
- The decomposition method (exact, taylor, krylov)
- The number of Trotter steps

In [6]:
# Initial program that OpenEvolve will optimize
def run_mp_trotter(hams, t, psi0):
    """
    Mixed precision Trotter evolution.
    
    Args:
        hams: List of Hamiltonian matrices (with coefficients already applied)
        t: Total evolution time
        psi0: Initial state vector
    
    Returns:
        Final state after evolution
    """
    from trovolut.backends.gpu_mp_emulator.dynamics import (
        Precision, PrecisionConfig, ExpTerm, TrotterSimulator,
        SecondOrderTrotter, PrecisionCaster
    )
    import numpy as np
    
    # Default: use FP32 for all terms with second-order Trotter
    n_steps = 100
    
    # Create ExpTerm objects with initial precision configuration
    terms = []
    for i, H in enumerate(hams):
        # Start with a simple configuration: FP32 everywhere
        precision = PrecisionConfig(
            compute=Precision.FP32,
            storage=Precision.FP32,
            accumulate=Precision.FP64
        )
        
        term = ExpTerm(
            name=f"term_{i}",
            generator=H,
            coefficient=1.0,  # Already in H
            decomposition="exact",
            precision=precision
        )
        terms.append(term)
    
    # Create simulator with second-order Trotter
    simulator = TrotterSimulator(
        terms=terms,
        formula=SecondOrderTrotter(),
        n_steps=n_steps,
        caster=PrecisionCaster(use_torch=False)
    )
    
    # Run evolution
    final_state, stats = simulator.evolve(psi0, t, track_stats=False)
    
    return final_state

## 6. Generate Files for OpenEvolve

Create the initial program and evaluator files.

In [7]:
# Create output directory
Path("mp_opt").mkdir(exist_ok=True)

# Build workflow for initial program and evaluator
workflow = trv.filemaker.Workflow(
    [trv.filemaker.EVO_START, run_mp_trotter, trv.filemaker.EVO_END],
    out_path="mp_opt/initial_program.py",
    imports=[
        "import scipy",
        "import numpy as np",
    ],
)

# Attach constraints to workflow (enables build_evaluator)
workflow = constraints(workflow)

# Build both initial program and evaluator
init_path = workflow.build()
evaluator_path = workflow.build_evaluator()

print(f"\nGenerated files:")
print(f"  Initial program: {init_path}")
print(f"  Evaluator:       {evaluator_path}")


Generated files:
  Initial program: mp_opt/initial_program.py
  Evaluator:       mp_opt/evaluator.py


## 7. Test the Initial Program

Verify that the initial implementation works and satisfies constraints.

In [8]:
# Load and test the evaluator
exec(open(evaluator_path).read())

result = evaluate(str(init_path))
print(f"\nInitial Program Evaluation:")
print(f"  Combined Score: {result['combined_score']:.4f}")
print(f"  All Satisfied:  {result['all_satisfied']}")
print(f"\nDetails:")
for name, info in result['details'].items():
    status = "✓" if info['satisfied'] else "✗"
    print(f"  {status} {name:10s}: {info['value']:.6e} (score: {info['score']:.4f})")

converged SCF energy = -864.654938666712

Initial Program Evaluation:
  Combined Score: 0.0000
  All Satisfied:  True

Details:
  ✗ flops     : inf (score: 0.0000)
  ✓ error     : 1.354693e-05 (score: 1.0000)


## 8. Run OpenEvolve Optimization

Now we'll use OpenEvolve to optimize the mixed precision Trotter implementation.

**What OpenEvolve will try:**
- Adjusting precisions (e.g., use FP16 for some terms, FP64 for critical ones)
- Changing Trotter formulas (first-order, second-order, higher-order)
- Optimizing the number of steps
- Using different decomposition methods (taylor, krylov) with optimized parameters
- Reordering terms to minimize error accumulation

In [9]:
import os

# Set your NVIDIA API key
# os.environ["NVIDIA_API_KEY"] = "your-api-key-here"

# Display the optimization prompt
print("OpenEvolve Optimization Prompt:")
print("=" * 70)
print(constraints.generate_prompt())
print("=" * 70)

OpenEvolve Optimization Prompt:
You are optimizing quantum simulation code. Your objectives are:

**Hard requirements (MUST satisfy):**
- Simulation error MUST be below 0.01 (hard requirement).

**Optimization goals (weighted):**
- Minimize FLOPs (target: <1.00e+12, weight: 2.0).


In [ ]:
from tabnanny import verbose
import nest_asyncio
import os
nest_asyncio.apply()

from openevolve import run_evolution
from openevolve.config import Config, LLMModelConfig, PromptConfig

config = Config(checkpoint_interval=10)
config.prompt = PromptConfig(system_message=constraints.generate_prompt())

# Configure NVIDIA inference endpoint
config.llm.models = [
    LLMModelConfig(
        name="openai/openai/gpt-5.2",
        api_base="https://inference-api.nvidia.com/v1",
        api_key=os.getenv("NVIDIA_API_KEY"),  # or set directly: api_key="your_key_here"
        temperature=0.7,
        max_tokens=1024
    )
]

result = run_evolution(
    initial_program=str(init_path),
    evaluator=str(evaluator_path),
    iterations=100,
    config=config,
    output_dir="opt/evolution_results",
)

2026-01-30 16:45:36,892 - INFO - Logging to opt/evolution_results/logs/openevolve_20260130_164536.log
2026-01-30 16:45:36,892 - INFO - Set random seed to 42 for reproducibility
2026-01-30 16:45:36,936 - INFO - Initialized OpenAI LLM with model: openai/openai/gpt-5.2
2026-01-30 16:45:36,936 - INFO - Initialized LLM ensemble with models: openai/openai/gpt-5.2 (weight: 1.00)
2026-01-30 16:45:36,942 - INFO - Initialized prompt sampler
2026-01-30 16:45:36,943 - INFO - Set custom templates: system=evaluator_system_message, user=None
2026-01-30 16:45:36,943 - INFO - Initialized program database with 0 programs
2026-01-30 16:45:36,947 - INFO - Successfully loaded evaluation function from mp_opt/evaluator.py
2026-01-30 16:45:36,947 - WARNING - Configuration has 'cascade_evaluation: true' but evaluator 'mp_opt/evaluator.py' does not define 'evaluate_stage1' function. This will fall back to direct evaluation, making the cascade setting useless. Consider setting 'cascade_evaluation: false' or impl

## 9. Analyze Results

Compare the initial and optimized implementations.

In [ ]:
# Reload evaluator to get latest results
exec(open(evaluator_path).read())

# Evaluate best program
best_result = evaluate(best_program)

print(f"\n{'='*70}")
print(f"COMPARISON: Initial vs. Optimized")
print(f"{'='*70}\n")

print(f"{'Metric':<20} {'Initial':>15} {'Optimized':>15} {'Improvement':>15}")
print("-" * 70)

# Combined score
initial_score = result['combined_score']
best_score = best_result['combined_score']
improvement = ((best_score - initial_score) / abs(initial_score)) * 100 if initial_score != 0 else 0
print(f"{'Combined Score':<20} {initial_score:>15.4f} {best_score:>15.4f} {improvement:>14.1f}%")

# Individual metrics
for metric in result['details'].keys():
    initial_val = result['details'][metric]['value']
    best_val = best_result['details'][metric]['value']
    
    # Calculate improvement (lower is better for cost/error)
    if initial_val != 0:
        improvement = ((initial_val - best_val) / abs(initial_val)) * 100
    else:
        improvement = 0
    
    print(f"{metric:<20} {initial_val:>15.6e} {best_val:>15.6e} {improvement:>14.1f}%")

print(f"\nSatisfied Constraints:")
print(f"  Initial:   {result['all_satisfied']}")
print(f"  Optimized: {best_result['all_satisfied']}")

## 10. Visualize Evolution Progress

Plot how the optimization progressed over iterations.

In [ ]:
import matplotlib.pyplot as plt
import json

# Load evolution history
log_dir = Path("mp_opt/evolution_results/logs")
if log_dir.exists():
    # Find the latest log file
    log_files = sorted(log_dir.glob("*.log"))
    if log_files:
        print(f"Found {len(log_files)} log files")
        
        # Parse log for scores and metrics
        iterations = []
        scores = []
        errors = []
        costs = []
        
        with open(log_files[-1], 'r') as f:
            for line in f:
                if "combined_score=" in line and "Iteration" in line:
                    # Extract iteration number and score
                    parts = line.split()
                    for i, part in enumerate(parts):
                        if part.startswith("Iteration") and i+1 < len(parts):
                            try:
                                iter_num = int(parts[i+1].rstrip(':'))
                                iterations.append(iter_num)
                            except:
                                pass
                        if "combined_score=" in part:
                            try:
                                score = float(part.split('=')[1].rstrip(','))
                                scores.append(score)
                            except:
                                pass
        
        if len(iterations) > 0 and len(scores) > 0:
            # Plot evolution
            fig, ax = plt.subplots(figsize=(10, 6))
            ax.plot(iterations[:len(scores)], scores, 'o-', linewidth=2, markersize=4)
            ax.set_xlabel('Iteration', fontsize=12)
            ax.set_ylabel('Combined Score', fontsize=12)
            ax.set_title('OpenEvolve Optimization Progress', fontsize=14, fontweight='bold')
            ax.grid(True, alpha=0.3)
            ax.axhline(y=initial_score, color='r', linestyle='--', label='Initial Score', alpha=0.7)
            ax.legend()
            plt.tight_layout()
            plt.savefig('mp_opt/evolution_progress.png', dpi=150)
            plt.show()
            
            print(f"\nEvolution statistics:")
            print(f"  Total iterations: {len(scores)}")
            print(f"  Best score: {max(scores):.4f} (iteration {iterations[scores.index(max(scores))]})")
            print(f"  Initial score: {initial_score:.4f}")
            print(f"  Improvement: {((max(scores) - initial_score) / abs(initial_score)) * 100:.1f}%")
        else:
            print("Could not parse enough data from log files")
else:
    print(f"Log directory not found: {log_dir}")

## 11. Inspect the Optimized Code

Let's look at what OpenEvolve changed in the implementation.

In [ ]:
# Display the optimized code
print("Optimized Implementation:")
print("=" * 70)
with open(best_program, 'r') as f:
    code = f.read()
    # Find the function definition
    start_idx = code.find('def run_mp_trotter')
    if start_idx != -1:
        # Print from function definition onwards (limit to reasonable length)
        print(code[start_idx:start_idx+2000])
        if len(code[start_idx:]) > 2000:
            print("\n... (code continues) ...")
    else:
        print(code[:2000])

## 12. Detailed Cost Analysis

Use the cost analyzer to understand the spacetime complexity of the optimized solution.

In [ ]:
# This section would require extracting the simulator configuration from the optimized code
# For demonstration, we show what kind of analysis is possible

print("""\nCost Analysis:\n
The optimized solution likely includes:

1. **Precision Optimizations:**
   - Small coefficients → FP16 (2x faster on GPU)
   - Large coefficients → FP32 or FP64 (better accuracy)
   - Critical terms → FP64 (numerical stability)

2. **Decomposition Method Selection:**
   - Diagonal terms → exact (cheap)
   - Dense terms with small norm → taylor (faster than expm)
   - Large dense terms → krylov (subspace methods)

3. **Trotter Formula Optimization:**
   - Balance between accuracy (higher order) and cost (more applications)
   - Custom orderings to minimize error accumulation

4. **Step Count Tuning:**
   - Fewer steps = less cost
   - But must maintain target accuracy
""")

## 13. Pareto Frontier Analysis (Optional)

Run OpenEvolve multiple times with different constraint settings to explore the accuracy-cost tradeoff.

In [ ]:
print("""\nTo generate a Pareto frontier:

1. Run OpenEvolve with different error thresholds:
   - max_error = 0.1 (loose)
   - max_error = 0.01 (moderate) 
   - max_error = 0.001 (tight)
   - max_error = 0.0001 (very tight)

2. For each setting, OpenEvolve will find the minimum-cost solution
   that satisfies the error constraint.

3. Plot error vs. FLOPs to visualize the tradeoff.

This reveals:
- Where diminishing returns begin (steep part of curve)
- The "sweet spot" for your application
- How much accuracy costs in computational resources
""")

## Summary

This notebook demonstrated:

1. **Mixed Precision Simulation**: Using FP16/32/64 selectively for different Pauli terms in a molecular Hamiltonian
2. **OpenEvolve Optimization**: Automatically finding the best precision configuration and Trotter splitting
3. **Multi-objective Optimization**: Minimizing FLOPs while maintaining accuracy
4. **Molecular Hamiltonian Evolution**: Applied to DMDS (dimethyl disulfide) with 4 qubits

**Key Insights:**
- Not all Pauli terms need high precision (small coefficients can use FP16)
- Decomposition method matters (exact vs taylor vs krylov)
- Trotter formula selection involves accuracy-cost tradeoffs
- OpenEvolve can discover non-obvious optimizations for molecular systems

**Next Steps:**
- Try larger molecular systems (more qubits, larger active spaces)
- Experiment with different molecules (H2O, CH4, etc.)
- Generate Pareto frontiers for your specific use case
- Profile GPU performance with optimized code
- Test on excited state dynamics or time-dependent Hamiltonians